In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import roc_curve, roc_auc_score, auc

In [2]:
# load data
df = pd.read_excel("../data/Base_coronavirus_31-05-2021.xlsx")
df.head()

,id,ano,semana,fecha_not,diresa,red,microred,establecimiento,institucion,clasificacion,...,prueba_rap,resultado_rap,fecha_res_rap,fecha_rap1,muestra_rap1,prueba_rap1,resultado_rap1,fecha_res_rap1,secuenciamiento,asintomatico
0,542,2021,5,09-02-2021,PUNO,PUNO,SIN MICRORED,"HOSP. REG. ""MANUEL NUÑEZ BUTRÓN"" - PUNO",GOBIERNO REGIONAL,DESCARTADO,...,NaN,NaN,NaN,00-00-0000,NaN,NaN,NaN,NaN,NaN,NaN
1,1234,2020,10,07-03-2020,PUNO,SAN ROMAN,SIN MICRORED,"HOSP. ""CARLOS MONGE MEDRANO"" - JULIACA",GOBIERNO REGIONAL,DESCARTADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1258,2020,10,09-03-2020,PUNO,PUNO,SIN MICRORED,HOSPITAL III DE ESSALUD,ESSALUD,DESCARTADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1266,2020,10,09-03-2020,PUNO,SAN ROMAN,SIN MICRORED,"HOSP. ""CARLOS MONGE MEDRANO"" - JULIACA",GOBIERNO REGIONAL,DESCARTADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1292,2021,9,05-03-2021,PUNO,PUNO,METROPOLITANO,C.S. METROPOLITANO PUNO,GOBIERNO REGIONAL,CONFIRMADO,...,SEROLOGIA,NEGATIVO,16-07-2020,00-00-0000,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Contar los valores únicos en la columna 'evolucion'
evolucion_counts = df['evolucion'].value_counts(dropna=False)
print("Valores únicos en la columna 'evolucion':\n", evolucion_counts)

Valores únicos en la columna 'evolucion':
 evolucion
NaN              67477
RECUPERADO       18943
ESTACIONARIA      5480
FALLECIÓ          2315
NO RECUPERADO      627
DESCONOCIDO        473
Name: count, dtype: int64


In [ ]:
# Crear la variable objetivo binaria 'es_fallecido'
# 1 si 'evolucion' es 'FALLECIî', 0 en caso contrario.
# vamos a limpiar los espacios en blanco y arreglar la codificacion del caracter faLLECIÎ.
df['evolucion_cleaned'] = df['evolucion'].str.strip().str.upper()

df['es_fallecido'] = (df['evolucion_cleaned'] == 'FALLECIÎ').astype(int)

# Aplicar one-hot encoding a 'hospitalizado' y asegurar que todas las características sean numéricas
df_processed = pd.get_dummies(df, columns=['hospitalizado'], prefix='hospitalizado', dummy_na=False)

# Definir las características para X.
features_base = ['edad', 'ano', 'semana']
hospitalizado_cols = [col for col in df_processed.columns if col.startswith('hospitalizado_')]

# Asegurarse de que no haya duplicados y que todas las columnas existan
features = features_base + hospitalizado_cols

# Seleccionar X y y del DataFrame procesado
X = df_processed[features].copy()
y = df_processed['es_fallecido'].copy()

# Combinar X e y para una limpieza consistente de valores nulos
data_logistic = pd.concat([X, y], axis=1)

# Eliminar filas donde la variable objetivo 'es_fallecido' es NaN (aunque ya la convertimos a int, puede haber NaNs en las features)
data_logistic = data_logistic.dropna(subset=['es_fallecido'])

# Eliminar filas donde las características seleccionadas tienen valores nulos
data_logistic = data_logistic.dropna(subset=features)

X_clean_logistic = data_logistic[features]
y_clean_logistic = data_logistic['es_fallecido']

print(f"Dimensiones de los datos después de la limpieza para regresión logística: {X_clean_logistic.shape}")
print("Conteo de la variable objetivo 'es_fallecido' después de la limpieza:")
print(y_clean_logistic.value_counts())
display(data_logistic.head())

Dimensiones de los datos después de la limpieza para regresión logística: (95315, 6)
Conteo de la variable objetivo 'es_fallecido' después de la limpieza:
es_fallecido
0    95315
Name: count, dtype: int64


,edad,ano,semana,hospitalizado_DESCONOCIDO,hospitalizado_NO,hospitalizado_SI,es_fallecido
0,31,2021,5,False,True,False,0
1,53,2020,10,True,False,False,0
2,47,2020,10,False,True,False,0
3,76,2020,10,False,True,False,0
4,49,2021,9,False,True,False,0


In [5]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_clean_logistic, y_clean_logistic, test_size=0.2, random_state=42, stratify=y_clean_logistic)

print(f"Tamaño del conjunto de entrenamiento (X_train_log): {X_train_log.shape}")
print(f"Tamaño del conjunto de prueba (X_test_log): {X_test_log.shape}")
print("Conteo de la variable objetivo en el conjunto de entrenamiento:")
print(y_train_log.value_counts(normalize=True))
print("Conteo de la variable objetivo en el conjunto de prueba:")
print(y_test_log.value_counts(normalize=True))

Tamaño del conjunto de entrenamiento (X_train_log): (76252, 6)
Tamaño del conjunto de prueba (X_test_log): (19063, 6)
Conteo de la variable objetivo en el conjunto de entrenamiento:
es_fallecido
0    1.0
Name: proportion, dtype: float64
Conteo de la variable objetivo en el conjunto de prueba:
es_fallecido
0    1.0
Name: proportion, dtype: float64


In [ ]:
# Escalado de características (importante para Regresión Logística con diferentes escalas)
scaler = StandardScaler()
X_train_scaled_log = scaler.fit_transform(X_train_log)
X_test_scaled_log = scaler.transform(X_test_log)

# Inicializar y entrenar el modelo de Regresión Logística
logistic_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' es bueno para datasets pequeños y binarios
logistic_model.fit(X_train_scaled_log, y_train_log)

print("Modelo de Regresión Logística entrenado.")
print(f"Coeficientes: {logistic_model.coef_}")
print(f"Intercepto: {logistic_model.intercept_}")

ARBOL DE DECISIONES

In [ ]:
# Entrenamos el modelo de Árbol de Decisiones
decision_tree_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
decision_tree_model.fit(X_train_scaled_log, y_train_log)

print("Modelo de Árbol de Decisiones entrenado.")

In [ ]:
# Realizar predicciones en el conjunto de prueba con el Árbol de Decisiones
y_pred_dt = decision_tree_model.predict(X_test_scaled_log)
y_pred_proba_dt = decision_tree_model.predict_proba(X_test_scaled_log)[:, 1]

# Evaluar el modelo de Árbol de Decisiones
accuracy_dt = accuracy_score(y_test_log, y_pred_dt)
precision_dt = precision_score(y_test_log, y_pred_dt)
recall_dt = recall_score(y_test_log, y_pred_dt)
f1_dt = f1_score(y_test_log, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test_log, y_pred_proba_dt)

print(f"\nÁrbol de Decisiones - Accuracy: {accuracy_dt:.2f}")
print(f"Árbol de Decisiones - Precision: {precision_dt:.2f}")
print(f"Árbol de Decisiones - Recall: {recall_dt:.2f}")
print(f"Árbol de Decisiones - F1-Score: {f1_dt:.2f}")
print(f"Árbol de Decisiones - ROC AUC Score: {roc_auc_dt:.2f}")

print("\nÁrbol de Decisiones - Classification Report:")
print(classification_report(y_test_log, y_pred_dt))

print("\nÁrbol de Decisiones - Confusion Matrix:")
conf_matrix_dt = confusion_matrix(y_test_log, y_pred_dt)
sns.heatmap(conf_matrix_dt, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Decision Tree)')
plt.show()

# Curva ROC para el Árbol de Decisiones
fpr_dt, tpr_dt, thresholds_dt = roc_curve(y_test_log, y_pred_proba_dt)
plt.figure(figsize=(8, 6))
plt.plot(fpr_dt, tpr_dt, color='blue', lw=2, label=f'ROC curve (area = {roc_auc_dt:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve (Decision Tree)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()